# 🏰 Lakehouse Storage Layer with DuckLake & DuckDB Catalog

This notebook provides a hands-on guide to **DuckLake** as a modern Lakehouse storage layer using **DuckDB** for both compute engine and metadata catalog (`hr_lake`).

--- 

## 🎯 Objectives & Core Capabilities
- **Lakehouse Catalog (`hr_lake`) & Decoupled Storage**: Store metadata in `lakehouse/hr_lake_catalog.db` and raw data files as Parquet in `lakehouse/hr_lake_data/`.
- **Transactions (ACID Atomicity & Consistency)**: Execute multi-statement writes safely with atomic commits and rollbacks during payroll processing.
- **Time Travel**: Query past versions and historical snapshots of tables without restoring backups.
- **Schema Evolution**: Modify table structures in-place (e.g., adding a performance rating column) without rewriting existing Parquet data files.

--- 

## 📌 DuckDB Engine vs. DuckLake Storage Layer
- **DuckDB Engine**: An in-process analytical SQL query engine responsible for parsing SQL, planning execution, and processing data in memory.
- **DuckLake Storage Layer (`hr_lake`)**: An open Lakehouse storage specification and extension for DuckDB that manages metadata catalogs, ACID transactions, and decoupled Parquet data storage.

## 🛠️ Section 1: Setup and Prerequisites

### Installation Commands
```bash
pip install duckdb
```
### DuckLake Extension & DuckDB Catalog Configuration
- **Plugin Required**: `ducklake` (Lakehouse storage engine plugin).
- **Attach Connection String**: `ATTACH 'lakehouse/hr_lake_catalog.db' AS hr_lake (TYPE DUCKLAKE, DATA_PATH 'lakehouse/hr_lake_data/');`
- **`lakehouse/hr_lake_catalog.db`**: DuckDB catalog file managing table schemas, active snapshots, transactions, and file tracking.
- **`lakehouse/hr_lake_data/`**: Directory containing raw, immutable Parquet files.

In [ ]:
# -----------------------------------------------------
# Step 1A: Clean local environment for a fresh run
# -----------------------------------------------------
import os
import shutil
import duckdb

DATA_LAKE_DIR = "lakehouse"
CATALOG_DB = os.path.join(DATA_LAKE_DIR, "hr_lake_catalog.db")
DATA_DIRECTORY = os.path.join(DATA_LAKE_DIR, "hr_lake_data/")
SESSION_DB = os.path.join(DATA_LAKE_DIR, "local_session.duckdb")

# Clean up previous lakehouse folder if re-running
if os.path.exists(DATA_LAKE_DIR):
    shutil.rmtree(DATA_LAKE_DIR)
os.makedirs(DATA_LAKE_DIR, exist_ok=True)

print("🧹 Environment cleaned up successfully.")

In [ ]:
%%sql
-- -----------------------------------------------------
-- Step 1B: Initialize DuckLake Extension & Mount hr_lake Catalog
-- -----------------------------------------------------
INSTALL ducklake;
LOAD ducklake;

ATTACH 'lakehouse/hr_lake_catalog.db' AS hr_lake (
    TYPE DUCKLAKE,
    DATA_PATH 'lakehouse/hr_lake_data/'
);

USE hr_lake;

In [ ]:
%%sql
-- -----------------------------------------------------
-- Step 1C: Create Table & Insert Initial Employees
-- -----------------------------------------------------
CREATE TABLE employees (
    id INTEGER, 
    first_name VARCHAR, 
    last_name VARCHAR, 
    department VARCHAR, 
    salary INTEGER
);

INSERT INTO employees VALUES 
(1, 'Tariq', 'Al-Ali', 'Engineering', 32000),
(2, 'Fatima', 'Al-Zahra', 'Marketing', 24000);

SELECT * FROM employees;

## 🔄 Section 2: Transactions (ACID Multi-Statement Writes)

### Concept Explanation
- **Atomicity**: All SQL statements inside `BEGIN TRANSACTION` commit together or roll back cleanly if an error occurs.
- **Consistency**: Protects the catalog and storage layer against partial updates, ensuring corrupted or incomplete writes never become visible.

### Realistic Demonstration Scenarios
1. **Successful Transaction (`COMMIT`)**: Promotes Tariq Al-Ali and inserts a new employee, Omar Al-Farooq.
2. **Failed Transaction (`ROLLBACK`)**: Simulates a realistic payroll calculation error (division by zero during a batch bonus calculation). The exception triggers `ROLLBACK`, leaving the dataset completely unchanged.

In [ ]:
%%sql
-- -----------------------------------------------------
-- Step 2A: Successful ACID Transaction (Salary Update + Omar Al-Farooq Insertion)
-- -----------------------------------------------------
BEGIN TRANSACTION;
UPDATE employees SET salary = 35000 WHERE first_name = 'Tariq';
INSERT INTO employees VALUES (3, 'Omar', 'Al-Farooq', 'Engineering', 28000);
COMMIT;

SELECT * FROM employees;

In [ ]:
%%sql
-- -----------------------------------------------------
-- Step 2B: Realistic Transaction Failure (Payroll Division by Zero Error -> ROLLBACK)
-- -----------------------------------------------------
BEGIN TRANSACTION;
UPDATE employees SET salary = 38000 WHERE id = 1;

-- Division by zero error during bonus calculation causes automatic transaction rollback
UPDATE employees SET salary = salary + (10000 / 0) WHERE department = 'Engineering';
COMMIT;

## ⏱️ Section 3: Time Travel (Querying Historical Snapshots)

### Concept Explanation
- **Snapshot Ledger**: DuckLake tracks immutable snapshot IDs in `hr_lake_catalog.db` whenever a transaction commits.
- **Time Travel Queries**: Query previous versions of a table using `AT (VERSION => n)` without making file copies or restoring database backups.

### Demonstration Steps
1. Inspect the snapshot log using `ducklake_snapshots('hr_lake')`.
2. Query **VERSION 2** (the initial state containing Tariq & Fatima before Omar was added).
3. Compare with the current version.

In [ ]:
%%sql
-- -----------------------------------------------------
-- Step 3A: View Snapshot Metadata Ledger
-- -----------------------------------------------------
SELECT snapshot_id, snapshot_time, commit_message 
FROM ducklake_snapshots('hr_lake');

In [ ]:
%%sql
-- -----------------------------------------------------
-- Step 3B: Time Travel Query to VERSION 2 & Latest Version
-- -----------------------------------------------------
-- Query historical snapshot at VERSION 2 (Initial Tariq & Fatima state)
SELECT * FROM employees AT (VERSION => 2);

-- Query current state (Latest snapshot)
SELECT * FROM employees;

## 🧬 Section 4: Zero-Copy Schema Evolution

### Concept Explanation
- **In-Place Metadata Updates**: Adding a column (`ALTER TABLE ... ADD COLUMN`) updates the catalog schema ledger in `hr_lake_catalog.db` without rewriting existing raw Parquet data files in `hr_lake_data/`.
- **Backward Compatibility**: Existing records automatically display `NULL` / `NaN` for newly added columns until updated.

### Demonstration Steps
1. Add `performance_rating DOUBLE` column.
2. Insert a new record (`Layla Mahmoud`) containing all 6 columns.
3. Inspect the evolved table structure.

In [ ]:
%%sql
-- -----------------------------------------------------
-- Step 4: Evolve Schema & Add Layla Mahmoud
-- -----------------------------------------------------
ALTER TABLE employees ADD COLUMN performance_rating DOUBLE;

INSERT INTO employees VALUES 
(4, 'Layla', 'Mahmoud', 'HR', 30000, 4.9);

SELECT * FROM employees;

## 📁 Section 5: Verification & Local Workspace Structure

Inspect your workspace directory structure after running this notebook to observe the decoupled storage design:

- **`lakehouse/hr_lake_catalog.db`**: The DuckDB relational catalog database managing table schemas, active snapshots, transactions, and file tracking.
- **`lakehouse/hr_lake_data/`**: A folder containing raw, queryable, optimized **Parquet files** generated cleanly via DuckDB writes.

## 📊 Section 6: DuckDB Engine vs. DuckLake Storage Layer Summary

| Feature / Capability | DuckDB Engine | DuckLake Storage Layer (`hr_lake`) |
| :--- | :--- | :--- |
| **Primary Role** | SQL Parsing, Query Execution, In-Memory Computation | Catalog Ledger, Snapshot History, Parquet Data Layout |
| **Catalog Database** | Default DuckDB system catalog | Dedicated `lakehouse/hr_lake_catalog.db` metadata database |
| **Transactions** | Session-level in-memory ACID | Multi-file ACID commits logged to catalog ledger |
| **Time Travel** | Not available for standard table structures | Instant historical queries via `AT (VERSION => n)` |
| **Schema Evolution** | Session table schema changes | Zero-copy schema evolution without rewriting Parquet files |
| **Storage Layout** | Monolithic local `.duckdb` file | Decoupled metadata catalog (`lakehouse/hr_lake_catalog.db`) & folder storage (`lakehouse/hr_lake_data/`) |

## 🏁 Section 7: Key Takeaways & Conclusion

1. **DuckDB Catalog & Storage Decoupling**: DuckLake separates metadata management (`lakehouse/hr_lake_catalog.db`) from raw data storage (`lakehouse/hr_lake_data/`).
2. **ACID Transactions**: Standard SQL transactions (`BEGIN TRANSACTION`, `COMMIT`, `ROLLBACK`) ensure data integrity and rollback failed calculations cleanly.
3. **Time Travel**: Snapshot logs track every commit, allowing instant time-travel queries via `AT (VERSION => n)`.
4. **Zero-Copy Schema Evolution**: Table schemas evolve in-place with `ALTER TABLE` without rewriting existing Parquet data files.